# 03 · Conversation-level evaluation of GraphNLI

GraphNLI predicts toxicity per toot. To compare it with DeTox-Fed, which labels
conversations, the toot-level predictions on the held-out test set are aggregated
per conversation with the same rule DeTox-Fed uses for its labels:

> a conversation is toxic if its root toot is toxic, or if at least `num` replies
> **and** at least a fraction `frac` of the replies are toxic (toot score ≥ `threshold`).

Sections:

1. **Detoxify-derived labels**: reference labels from the Detoxify aggregates with
   the fixed rule (0.5, 1, 0.01).
2. **External labels** (e.g. LLM or human annotations): all rules of the grid,
   reporting the best rule and the fixed rule, mean ± std over training runs.
3. **Calibrated rule**: the external-labelled conversations are split into a
   calibration and a test set; the rule is chosen on calibration only.

Input: `final_test_predictions.csv` from one or more `2_FL_GraphNLI_model_training/train.py` runs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

## Configuration

In [ ]:
DATASET = "pleroma"  # or "mastodon"

DATA_DIR = Path("../../data") / DATASET
RESULTS_DIR = Path("../../results/graphnli") / DATASET
OUTPUT_DIR = RESULTS_DIR / "conversation_level"

# One predictions file per training run (e.g. one run per seed).
PREDICTION_FILES = sorted(RESULTS_DIR.glob("run_*/final_test_predictions.csv"))

TOOTS_CSV = DATA_DIR / f"{DATASET}_data.csv"                    # root_toot flag
DETOXIFY_LABELS_CSV = DATA_DIR / f"final_{DATASET}_nodes_conv_tox.csv"

# External conversation labels: instance_toot_found_id, conversation_id and a 0/1 label column.
EXTERNAL_LABELS_CSV = DATA_DIR / "annotations" / "holdout_test_set.csv"
EXTERNAL_LABEL_COL = "gpt_label"

FIXED_RULE = {"bin_threshold": 0.5, "conv_threshold_num": 1, "conv_threshold_fraction": 0.01}

BIN_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]
CONV_THRESHOLD_NUMS = [1, 2, 3, 4, 5]
CONV_THRESHOLD_FRACTIONS = [0.01, 0.05, 0.10, 0.20, 0.30]

CALIBRATION_SPLIT = {"pleroma": (700, 300), "mastodon": (300, 200)}[DATASET]  # (calibration, test) conversations
RANDOM_STATE = 42

KEY = ["instance_id", "conversation_id"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(PREDICTION_FILES)} prediction files")

## Helper functions

In [ ]:
def conversation_statistics(predictions, toots, thresholds=BIN_THRESHOLDS):
    """
    Per conversation: predicted root-toot score, and number / fraction of replies
    with a predicted score ≥ each threshold.
    """
    root_lookup = toots[["instance_toot_found_id", "conversation_id", "toot_id", "root_toot"]].drop_duplicates(
        subset=["instance_toot_found_id", "conversation_id", "toot_id"]
    )
    df = predictions.merge(
        root_lookup,
        how="left",
        left_on=["instance_id", "conversation_id", "toot_id"],
        right_on=["instance_toot_found_id", "conversation_id", "toot_id"],
        validate="many_to_one",
    ).drop(columns=["instance_toot_found_id"])

    if df["root_toot"].dtype != bool:
        df["root_toot"] = df["root_toot"].astype(str).str.lower().isin(["true", "1", "yes"])

    df["root_toxicity"] = np.where(df["root_toot"], df["predicted_score"], np.nan)
    conv = (
        df.groupby(KEY)
        .agg(root_toxicity=("root_toxicity", "max"), n_posts=("toot_id", "size"), n_roots=("root_toot", "sum"))
        .reset_index()
    )

    replies = df[~df["root_toot"]].copy()
    conv = conv.merge(replies.groupby(KEY).size().reset_index(name="n_replies"), on=KEY, how="left")
    conv["n_replies"] = conv["n_replies"].fillna(0).astype(int)

    for t in thresholds:
        suffix = str(t).replace(".", "_")
        replies[f"is_toxic_{suffix}"] = replies["predicted_score"] >= t
        stats = (
            replies.groupby(KEY)[f"is_toxic_{suffix}"]
            .agg(["sum", "mean"])
            .reset_index()
            .rename(columns={"sum": f"num_toxic_replies_{suffix}", "mean": f"frac_toxic_replies_{suffix}"})
        )
        conv = conv.merge(stats, on=KEY, how="left")
        conv[f"num_toxic_replies_{suffix}"] = conv[f"num_toxic_replies_{suffix}"].fillna(0).astype(int)
        conv[f"frac_toxic_replies_{suffix}"] = conv[f"frac_toxic_replies_{suffix}"].fillna(0.0)

    return conv


def apply_rule(conv, bin_threshold, conv_threshold_num, conv_threshold_fraction):
    suffix = str(bin_threshold).replace(".", "_")
    return (
        (conv["root_toxicity"] >= bin_threshold)
        | (
            (conv[f"num_toxic_replies_{suffix}"] >= conv_threshold_num)
            & (conv[f"frac_toxic_replies_{suffix}"] >= conv_threshold_fraction)
        )
    ).astype(int)


def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "test_total": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "toxic_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "toxic_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "toxic_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "predicted_toxic": int((np.asarray(y_pred) == 1).sum()),
    }


def evaluate_rule(conv, labels, label_col, rule):
    comparison = conv.merge(labels[KEY + [label_col]], on=KEY, how="inner", validate="one_to_one")
    comparison["predicted_label"] = apply_rule(comparison, **rule)
    return calculate_metrics(comparison[label_col].astype(int), comparison["predicted_label"]), comparison


def rule_grid(conv, labels, label_col):
    comparison = conv.merge(labels[KEY + [label_col]], on=KEY, how="inner")
    y_true = comparison[label_col].astype(int)
    rows = [
        {
            "bin_threshold": t,
            "conv_threshold_num": n,
            "conv_threshold_fraction": f,
            **calculate_metrics(y_true, apply_rule(comparison, t, n, f)),
        }
        for t in BIN_THRESHOLDS
        for n in CONV_THRESHOLD_NUMS
        for f in CONV_THRESHOLD_FRACTIONS
    ]
    return pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)


def mean_std(df, metrics=("macro_f1", "toxic_precision", "toxic_recall")):
    return pd.DataFrame(
        {m: [f"{df[m].mean():.4f} ± {df[m].std(ddof=1):.4f}"] for m in metrics}
    )

## Load

In [ ]:
toots = pd.read_csv(TOOTS_CSV, usecols=["instance_toot_found_id", "conversation_id", "toot_id", "root_toot"], low_memory=False)

runs = {}
for path in PREDICTION_FILES:
    predictions = pd.read_csv(path)
    runs[path.parent.name] = conversation_statistics(predictions, toots)
    print(f"{path.parent.name}: {len(predictions):,} toots, {len(runs[path.parent.name]):,} conversations")

## 1 · Detoxify-derived labels

Reference label of each test conversation from the Detoxify aggregates, with the
fixed rule; GraphNLI's conversation label uses the same rule on its predicted scores.

In [ ]:
detoxify = pd.read_csv(DETOXIFY_LABELS_CSV).rename(columns={"instance_toot_found_id": "instance_id"})
detoxify["true_label"] = apply_rule(detoxify, **FIXED_RULE)

detoxify_results = []
for run, conv in runs.items():
    metrics, predictions = evaluate_rule(conv, detoxify, "true_label", FIXED_RULE)
    predictions.to_csv(OUTPUT_DIR / f"detoxify_labels_predictions_{run}.csv", index=False)
    detoxify_results.append({"run": run, **metrics})

detoxify_results = pd.DataFrame(detoxify_results)
detoxify_results.to_csv(OUTPUT_DIR / "detoxify_labels_metrics.csv", index=False)
display(detoxify_results)
mean_std(detoxify_results)

## 2 · External labels: all rules

For each run, every rule of the grid is evaluated against the external labels.
Reported: the best rule per run (by macro-F1) and the fixed rule.

In [ ]:
external = pd.read_csv(EXTERNAL_LABELS_CSV).rename(columns={"instance_toot_found_id": "instance_id"})
external = external.drop_duplicates(subset=KEY)

best_rows, fixed_rows = [], []
for run, conv in runs.items():
    grid = rule_grid(conv, external, EXTERNAL_LABEL_COL)
    grid.to_csv(OUTPUT_DIR / f"external_labels_rule_grid_{run}.csv", index=False)

    best_rows.append({"run": run, **grid.iloc[0].to_dict()})
    fixed = grid[
        np.isclose(grid["bin_threshold"], FIXED_RULE["bin_threshold"])
        & np.isclose(grid["conv_threshold_num"], FIXED_RULE["conv_threshold_num"])
        & np.isclose(grid["conv_threshold_fraction"], FIXED_RULE["conv_threshold_fraction"])
    ]
    fixed_rows.append({"run": run, **fixed.iloc[0].to_dict()})

best_df, fixed_df = pd.DataFrame(best_rows), pd.DataFrame(fixed_rows)
best_df.to_csv(OUTPUT_DIR / "external_labels_best_rule.csv", index=False)
fixed_df.to_csv(OUTPUT_DIR / "external_labels_fixed_rule.csv", index=False)

summary = pd.concat(
    [mean_std(best_df).assign(rule="best macro-F1 per run"), mean_std(fixed_df).assign(rule="fixed (0.5, 1, 0.01)")],
    ignore_index=True,
)
summary.to_csv(OUTPUT_DIR / "external_labels_summary.csv", index=False)
summary

## 3 · External labels: calibrated rule

The external-labelled conversations present in the predictions are split
(stratified) into a calibration and a test set. The rule with the best macro-F1 on
the calibration set is then applied unchanged to the test set.

In [ ]:
calibration_size, test_size = CALIBRATION_SPLIT
calibrated_results = []

for run, conv in runs.items():
    labelled = external[KEY + [EXTERNAL_LABEL_COL]].merge(conv[KEY], on=KEY, how="inner", validate="one_to_one")
    if len(labelled) < calibration_size + test_size:
        raise ValueError(f"{run}: need {calibration_size + test_size} labelled conversations, found {len(labelled)}.")

    calibration, test = train_test_split(
        labelled,
        train_size=calibration_size,
        test_size=test_size,
        stratify=labelled[EXTERNAL_LABEL_COL],
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    grid = rule_grid(conv.merge(calibration[KEY], on=KEY), calibration, EXTERNAL_LABEL_COL)
    grid.to_csv(OUTPUT_DIR / f"calibration_rule_grid_{run}.csv", index=False)
    best_rule = {k: grid.iloc[0][k] for k in ("bin_threshold", "conv_threshold_num", "conv_threshold_fraction")}
    best_rule["conv_threshold_num"] = int(best_rule["conv_threshold_num"])

    metrics, predictions = evaluate_rule(conv.merge(test[KEY], on=KEY), test, EXTERNAL_LABEL_COL, best_rule)
    predictions.to_csv(OUTPUT_DIR / f"calibrated_rule_test_predictions_{run}.csv", index=False)
    calibrated_results.append({"run": run, **best_rule, **metrics})

calibrated_results = pd.DataFrame(calibrated_results)
calibrated_results.to_csv(OUTPUT_DIR / "calibrated_rule_metrics.csv", index=False)
display(calibrated_results)
mean_std(calibrated_results)